# Swahili news classification with sequential models

Runs the full project on Google Colab: data download, exploratory analysis, five models and the comparison.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Approximate times on a T4: EDA 5 min, baselines 30 min, BiLSTM 30 min, TextCNN 10 min, AfriBERTa 3 to 4 hours (four fine-tuning runs; finished runs are cached in `results/transformer_runs/`, so rerunning an interrupted cell skips them).
Every step writes to `results/` and `figures/`, and the repository already contains the outputs of our runs,
so you can skip any training cell and still run the comparison cells.

In [ ]:
import os

REPO_URL = "https://github.com/irachrist1/swahili-news-sequential-models.git"
REPO_DIR = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists("src"):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!pip install -q -r requirements.txt
%cd src

In [ ]:
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 120)

def show(name, width=800):
    display(Image(filename=f"../figures/{name}", width=width))

## 1. Data
Downloads the corpus (22,207 training and 7,338 test articles) and removes empty, duplicate and train/test-overlapping articles.

In [ ]:
!python data.py

## 2. Exploratory analysis

In [ ]:
!python eda.py > /dev/null
import json
summary = json.load(open('../results/eda_summary.json'))
{k: summary[k] for k in ['cleaning', 'imbalance_ratio', 'length_words', 'noise', 'afriberta_tokens']}

In [ ]:
for name in ['eda_class_distribution.png', 'eda_lengths.png', 'eda_vocabulary.png', 'eda_positional_information.png', 'eda_code_switching.png']:
    show(name)

In [ ]:
pd.read_csv('../results/eda_top_words.csv')

## 3. Baselines: TF-IDF with Naive Bayes and Logistic Regression

In [ ]:
!python baselines.py

## 4. BiLSTM with attention

In [ ]:
!python train_neural.py --model bilstm

In [ ]:
show('learning_curves_bilstm.png')

## 5. TextCNN

In [ ]:
!python train_neural.py --model textcnn

In [ ]:
show('learning_curves_textcnn.png')

## 6. AfriBERTa-small fine-tuning
The longest step. To check the setup first, run `!python train_transformer.py --quick` (1 epoch on 2,000 articles).

In [ ]:
!python train_transformer.py

In [ ]:
show('learning_curves_afriberta.png')

## 7. Experiment log
Every run, what it changed, and why.

In [ ]:
log = pd.read_csv('../results/experiment_log.csv')
log[['experiment_id', 'model', 'owner', 'change', 'val_macro_f1', 'test_macro_f1']]

In [ ]:
show('experiment_progression.png', 900)

## 8. Model comparison

In [ ]:
!python compare_models.py > /dev/null
pd.read_csv('../results/model_comparison.csv')

In [ ]:
pd.read_csv('../results/per_class_f1.csv', index_col=0)

In [ ]:
pd.read_csv('../results/significance.csv')

In [ ]:
for name in ['confusion_all_models.png', 'per_class_f1.png', 'roc_curves.png', 'calibration.png']:
    show(name, 1000 if name.startswith('confusion') else 650)

## 9. Error analysis

In [ ]:
!python error_analysis.py > /dev/null
show('error_by_length.png')
pd.read_csv('../results/error_top_confusions.csv')

In [ ]:
pd.read_csv('../results/error_hard_examples.csv')

In [ ]:
pd.read_csv('../results/error_bilstm_attention.csv')